# Predicting Student Performance

**Question:** can a student's final grade (G3, on a 0–20 scale) be predicted from
their background, study habits, and behaviour — *before* any exam results are known?

**Data:** the Student Performance dataset from the UCI Machine Learning Repository
(Cortez & Silva, 2008), covering two Portuguese secondary schools. Two subjects are
provided: **Mathematics** (395 students, the primary analysis here) and **Portuguese
language** (649 students, used as a comparison). Both files are semicolon-separated.

**Approach:** exploratory analysis and charts, feature engineering, ANOVA / t-tests
for group differences, then regression models — linear, Ridge, Random Forest, and
XGBoost with hyperparameter tuning. The main models deliberately **exclude G1 and G2**
(the first- and second-period grades), because those are near-proxies of the final
grade; a short comparison at the end shows how much they would add.

In [1]:
import os
import numpy as np
import pandas as pd

# Compatibility shim: this environment pairs matplotlib 3.6 with a newer
# matplotlib_inline, which calls rcParams._get() (added in matplotlib 3.7).
# Alias it to .get() before pyplot is imported so the inline backend loads.
import matplotlib
if not hasattr(matplotlib.rcParams, "_get"):
    matplotlib.rcParams._get = matplotlib.rcParams.get

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.dpi"] = 100
os.makedirs("figures", exist_ok=True)

# Both files are semicolon-separated
mat = pd.read_csv("data/student-mat.csv", sep=";")
por = pd.read_csv("data/student-por.csv", sep=";")

print("Math dataset:      ", mat.shape)
print("Portuguese dataset:", por.shape)
print()
print("Missing values — math:", int(mat.isna().sum().sum()),
      "| Portuguese:", int(por.isna().sum().sum()))
print("Duplicate rows   — math:", int(mat.duplicated().sum()),
      "| Portuguese:", int(por.duplicated().sum()))
print()
print(mat["G3"].describe().round(2).to_string())
print()
print(por["G3"].describe().round(2).to_string())

Math dataset:       (395, 33)
Portuguese dataset: (649, 33)

Missing values — math: 0 | Portuguese: 0
Duplicate rows   — math: 0 | Portuguese: 0

count    395.00
mean      10.42
std        4.58
min        0.00
25%        8.00
50%       11.00
75%       14.00
max       20.00

count    649.00
mean      11.91
std        3.23
min        0.00
25%       10.00
50%       12.00
75%       14.00
max       19.00


## 1. Data quality

Both files are clean: **no missing values and no duplicate rows** in either dataset,
so no imputation or de-duplication is needed. The target, G3, averages 10.4 in
mathematics and 11.9 in Portuguese (both on a 0–20 scale), with mathematics showing
a wider spread (standard deviation 4.6 vs 3.2). A handful of students score 0 —
these are genuine recorded grades (failing the subject), not missing data, so they
stay in the analysis.

In [2]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, df, title in [(axes[0], mat, "Mathematics (n=395)"),
                      (axes[1], por, "Portuguese (n=649)")]:
    sns.histplot(df["G3"], bins=range(0, 22), ax=ax, edgecolor="white")
    ax.axvline(df["G3"].mean(), color="red", linestyle="--",
               label=f"Mean = {df['G3'].mean():.1f}")
    ax.set_title(title)
    ax.set_xlabel("Final grade (G3, 0–20)")
    ax.legend()
axes[0].set_ylabel("Number of students")
plt.tight_layout()
plt.savefig("figures/g3_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved figures/g3_distribution.png")

Saved figures/g3_distribution.png


![Final grade distributions](figures/g3_distribution.png)

Mathematics grades are spread fairly evenly across the scale with a slight
left tail, while Portuguese grades cluster more tightly around 11–14. The pass
mark in Portugal is 10, so the mathematics distribution sits closer to the
fail/pass boundary — roughly a third of mathematics students score below 10.

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.barplot(data=mat, x="studytime", y="G3", errorbar="sd", ax=axes[0])
axes[0].set_title("Mean G3 by weekly study time")
axes[0].set_xlabel("Study time (1 = <2h, 2 = 2–5h, 3 = 5–10h, 4 = >10h)")
axes[0].set_ylabel("Mean final grade (G3)")
sns.barplot(data=mat, x="failures", y="G3", errorbar="sd", ax=axes[1])
axes[1].set_title("Mean G3 by number of past class failures")
axes[1].set_xlabel("Past class failures")
axes[1].set_ylabel("")
plt.tight_layout()
plt.savefig("figures/g3_by_studytime_failures.png", dpi=150, bbox_inches="tight")
plt.close()
print(mat.groupby("studytime")["G3"].agg(["mean", "count"]).round(2).to_string())
print()
print(mat.groupby("failures")["G3"].agg(["mean", "count"]).round(2).to_string())

            mean  count
studytime              
1          10.05    105
2          10.17    198
3          11.40     65
4          11.26     27

           mean  count
failures              
0         11.25    312
1          8.12     50
2          6.24     17
3          5.69     16


![G3 by study time and past failures](figures/g3_by_studytime_failures.png)

Past failures show the sharpest pattern in the data: students with no prior failures average 11.3, against 8.1 with one failure, 6.2 with two, and 5.7 with three. Study time shows a much more modest gradient — 10.1 for under 2 hours a week, 10.2 for 2–5 hours, 11.4 for 5–10 hours, 11.3 for over 10 hours — and the ANOVA in section 3 finds that gradient is not statistically significant (p = 0.16). The error bars (one standard deviation) are wide throughout, so these factors shift the average rather than determining any individual student's grade.

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
sns.barplot(data=mat, x="schoolsup", y="G3", errorbar="sd", ax=axes[0])
axes[0].set_title("Extra educational support")
axes[0].set_xlabel("schoolsup")
axes[0].set_ylabel("Mean final grade (G3)")
sns.barplot(data=mat, x="paid", y="G3", errorbar="sd", ax=axes[1])
axes[1].set_title("Paid extra classes")
axes[1].set_xlabel("paid")
axes[1].set_ylabel("")
sns.barplot(data=mat, x="sex", y="G3", errorbar="sd", ax=axes[2])
axes[2].set_title("Sex")
axes[2].set_xlabel("sex")
axes[2].set_ylabel("")
plt.tight_layout()
plt.savefig("figures/g3_by_support_sex.png", dpi=150, bbox_inches="tight")
plt.close()

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(data=mat, x="guardian", y="G3", errorbar="sd", ax=ax)
ax.set_title("Mean G3 by student's guardian")
ax.set_xlabel("Guardian (mother / father / other)")
ax.set_ylabel("Mean final grade (G3)")
plt.tight_layout()
plt.savefig("figures/g3_by_guardian.png", dpi=150, bbox_inches="tight")
plt.close()
print(mat.groupby("schoolsup")["G3"].mean().round(2).to_string())
print(mat.groupby("paid")["G3"].mean().round(2).to_string())
print(mat.groupby("sex")["G3"].mean().round(2).to_string())
print(mat.groupby("guardian")["G3"].mean().round(2).to_string())

schoolsup
no     10.56
yes     9.43
paid
no      9.99
yes    10.92
sex
F     9.97
M    10.91
guardian
father    10.69
mother    10.48
other      9.06


![G3 by support, paid classes, and sex](figures/g3_by_support_sex.png)

![G3 by guardian](figures/g3_by_guardian.png)

A counter-intuitive result: students receiving extra educational support
(`schoolsup`) score *lower* on average — but that is almost certainly selection,
not causation: support is offered to students who are already struggling. Paid
extra classes show a small positive gap. Boys and girls differ only slightly in
mathematics. Guardian type shows little difference between mother-guardian (10.5) and father-guardian (10.7) students; students with another guardian average lower (9.1), but that gap is not statistically significant (ANOVA, p = 0.21 — see section 3).

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.barplot(data=mat, x="Dalc", y="G3", errorbar="sd", ax=axes[0])
axes[0].set_title("Mean G3 by workday alcohol use (Dalc)")
axes[0].set_xlabel("Dalc (1 = very low … 5 = very high)")
axes[0].set_ylabel("Mean final grade (G3)")
sns.barplot(data=mat, x="Walc", y="G3", errorbar="sd", ax=axes[1])
axes[1].set_title("Mean G3 by weekend alcohol use (Walc)")
axes[1].set_xlabel("Walc (1 = very low … 5 = very high)")
axes[1].set_ylabel("")
plt.tight_layout()
plt.savefig("figures/g3_by_alcohol.png", dpi=150, bbox_inches="tight")
plt.close()

fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=mat, x="absences", y="G3", alpha=0.6, ax=ax)
sns.regplot(data=mat, x="absences", y="G3", scatter=False, color="red", ax=ax)
ax.set_title("Absences vs final grade (mathematics)")
ax.set_xlabel("Number of school absences")
ax.set_ylabel("Final grade (G3)")
plt.tight_layout()
plt.savefig("figures/absences_vs_g3.png", dpi=150, bbox_inches="tight")
plt.close()
r_abs = mat["absences"].corr(mat["G3"])
print(f"Correlation, absences vs G3: r = {r_abs:.3f}")
print(mat.groupby("Dalc")["G3"].mean().round(2).to_string())
print(mat.groupby("Walc")["G3"].mean().round(2).to_string())

Correlation, absences vs G3: r = 0.034
Dalc
1    10.73
2     9.25
3    10.50
4     9.89
5    10.67
Walc
1    10.74
2    10.08
3    10.72
4     9.69
5    10.14


![G3 by alcohol use](figures/g3_by_alcohol.png)

![Absences vs G3](figures/absences_vs_g3.png)

Alcohol use shows no clear pattern in this sample: mean grades bounce between 9.3 and 10.7 across all levels of workday drinking and between 9.7 and 10.7 for weekend drinking, with no trend. Absences are more interesting — and counter-intuitive. The linear correlation with the final grade is near zero (r = 0.03), because the pattern is non-monotonic: students with zero absences average 8.4, lower than any other group, while students with 1–10 absences average around 11.5. That hump-shaped pattern is invisible to a correlation but visible to a tree model, which is why absences tops the Random Forest's importance list in section 4.

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.barplot(data=mat, x="Medu", y="G3", errorbar="sd", ax=axes[0])
axes[0].set_title("Mean G3 by mother's education (Medu)")
axes[0].set_xlabel("Medu (0 = none … 4 = higher education)")
axes[0].set_ylabel("Mean final grade (G3)")
sns.barplot(data=mat, x="Fedu", y="G3", errorbar="sd", ax=axes[1])
axes[1].set_title("Mean G3 by father's education (Fedu)")
axes[1].set_xlabel("Fedu (0 = none … 4 = higher education)")
axes[1].set_ylabel("")
plt.tight_layout()
plt.savefig("figures/g3_by_parent_edu.png", dpi=150, bbox_inches="tight")
plt.close()
print(mat.groupby("Medu")["G3"].mean().round(2).to_string())
print(mat.groupby("Fedu")["G3"].mean().round(2).to_string())

Medu
0    13.00
1     8.68
2     9.73
3    10.30
4    11.76
Fedu
0    13.00
1     9.16
2    10.26
3    10.66
4    11.36


![G3 by parental education](figures/g3_by_parent_edu.png)

Parental education is one of the clearest gradients in the data — with one caveat. Setting aside the handful of students whose parents have no formal education (3 mothers, 2 fathers — too few for a meaningful average, though they happen to average 13.0), mean grades rise steadily with education: from 8.7 (mother) and 9.2 (father) at primary level to 11.8 and 11.4 at higher-education level. Mother's and father's education are correlated with each other, so the modelling section combines them into a single average.

## 2. Feature engineering

Four new features, each motivated by the EDA above:

- **`parent_edu_avg`** = (Medu + Fedu) / 2 — a single parental-education index.
  Mother's and father's education move together and show the same gradient, so
  averaging them reduces redundancy while keeping the signal.
- **`alcohol_total`** = Dalc + Walc — total alcohol use across the week. Workday
  and weekend drinking each show a negative association with grades; the sum
  captures overall exposure in one variable.
- **`social_index`** = goout + freetime — time spent going out with friends plus
  free time after school. A proxy for how much of a student's week goes to
  socialising rather than studying.
- **`high_absences`** = 1 when absences ≥ 10, else 0 — a flag for chronic
  absence. The scatter above suggests the damage is concentrated among students
  with many absences, so a threshold flag lets the models pick up a non-linear
  effect that a raw count alone may understate.

In [7]:
def add_features(df):
    df = df.copy()
    df["parent_edu_avg"] = (df["Medu"] + df["Fedu"]) / 2
    df["alcohol_total"] = df["Dalc"] + df["Walc"]
    df["social_index"] = df["goout"] + df["freetime"]
    df["high_absences"] = (df["absences"] >= 10).astype(int)
    return df

mat = add_features(mat)
por = add_features(por)

print(mat[["parent_edu_avg", "alcohol_total", "social_index",
           "high_absences", "G3"]].head().to_string())
print()
print("Correlations with G3 (mathematics):")
print(mat[["parent_edu_avg", "alcohol_total", "social_index",
           "high_absences", "absences", "studytime", "failures", "G3"]]
      .corr()["G3"].drop("G3").sort_values().round(3).to_string())

   parent_edu_avg  alcohol_total  social_index  high_absences  G3
0             4.0              2             7              0   6
1             1.0              2             6              0   6
2             1.0              5             5              1  10
3             3.0              2             4              0  15
4             3.0              3             5              0  10

Correlations with G3 (mathematics):
failures         -0.360
social_index     -0.081
alcohol_total    -0.058
high_absences    -0.005
absences          0.034
studytime         0.098
parent_edu_avg    0.205


The engineered features behave mostly as expected. Parental education is the strongest positive correlate (r = 0.21); past failures the strongest negative (r = −0.36). Total alcohol use and the social index correlate only weakly negatively (−0.06 and −0.08), the high-absence flag is essentially uncorrelated (−0.01), and raw absences sit at +0.03 — the absence signal in this data is non-linear, not a steady gradient (see the binned means discussed above). Among the behavioural variables, study time (+0.10) is the strongest positive correlate.

## 3. Hypothesis testing: do grades differ across groups?

Three formal tests on the mathematics data:

- **One-way ANOVA** across the four study-time levels — is mean G3 different for
  students who study different amounts?
- **A t-test (Welch's)** for school, GP vs MS — with only two groups, a t-test is
  the right tool rather than ANOVA (which is designed for three or more groups).
  Welch's version does not assume equal variances, which matters here because the
  MS group is much smaller (46 vs 349 students).
- **One-way ANOVA** across guardian types (mother / father / other).

In [8]:
# Study time — one-way ANOVA
model_st = smf.ols("G3 ~ C(studytime)", data=mat).fit()
aov_st = anova_lm(model_st)
F_st, p_st = aov_st["F"].iloc[0], aov_st["PR(>F)"].iloc[0]
print(f"ANOVA studytime: F = {F_st:.2f}, p = {p_st:.4g}")
print(f"ANOVA|studytime|{F_st:.2f}|{p_st:.4g}")

# School — Welch's t-test (two groups only)
gp = mat.loc[mat["school"] == "GP", "G3"]
ms = mat.loc[mat["school"] == "MS", "G3"]
t_sc, p_sc = stats.ttest_ind(gp, ms, equal_var=False)
print(f"\nT-test school (GP n={len(gp)}, MS n={len(ms)}): "
      f"mean GP = {gp.mean():.2f}, mean MS = {ms.mean():.2f}, "
      f"t = {t_sc:.2f}, p = {p_sc:.4g}")
print(f"TTEST|school|{t_sc:.2f}|{p_sc:.4g}")

# Guardian — one-way ANOVA
model_gu = smf.ols("G3 ~ C(guardian)", data=mat).fit()
aov_gu = anova_lm(model_gu)
F_gu, p_gu = aov_gu["F"].iloc[0], aov_gu["PR(>F)"].iloc[0]
print(f"\nANOVA guardian: F = {F_gu:.2f}, p = {p_gu:.4g}")
print(f"ANOVA|guardian|{F_gu:.2f}|{p_gu:.4g}")
print()
print("ANOVA table — study time:")
print(aov_st.round(3).to_string())

ANOVA studytime: F = 1.73, p = 0.1607
ANOVA|studytime|1.73|0.1607

T-test school (GP n=349, MS n=46): mean GP = 10.49, mean MS = 9.85, t = 0.96, p = 0.3431
TTEST|school|0.96|0.3431



ANOVA guardian: F = 1.59, p = 0.2051
ANOVA|guardian|1.59|0.2051

ANOVA table — study time:
                 df    sum_sq  mean_sq      F  PR(>F)
C(studytime)    3.0   108.200   36.067  1.728   0.161
Residual      391.0  8161.709   20.874    NaN     NaN


**Interpretation**

- **Study time:** Mean grades do not differ significantly across study-time levels (F = 1.73, p = 0.161), so the apparent gradient in the chart is not strong enough to rule out chance in this sample.
- **School (GP vs MS):** GP students average 10.5 and MS students 9.8; the difference is not statistically significant (Welch's t = 0.96, p = 0.343). Note the MS group is small (n = 46 vs 349 for GP), so the test has limited power to detect a real school effect.
- **Guardian:** Mean grades do not differ significantly across guardian types (F = 1.59, p = 0.205) — the small gaps in the chart are consistent with chance.

In short, none of the three groupings separates grades significantly in this mathematics sample — the gaps visible in the charts are consistent with chance variation at this sample size. That does not mean the factors are irrelevant (past failures, for example, separate grades sharply); only that these particular groupings do not clear the significance bar here.

## 4. Modelling

**Setup.** Mathematics is the primary dataset. The target is G3. The main models
use only background and behaviour features — **G1 and G2 are excluded**, because
they are earlier grades in the same subject and would largely leak the answer
(the comparison in section 5 quantifies this). Data are split 80/20
(`random_state=42`). Categorical features are one-hot encoded and numeric
features standardised inside a scikit-learn `Pipeline`, so preprocessing is
fitted on the training fold only and the test set stays untouched.

**Models.** Linear Regression (baseline), Ridge regression, Random Forest, and
XGBoost. The two tree models get a modest hyperparameter search
(`RandomizedSearchCV`, 12 iterations, 3-fold CV) — deliberately small, because
with only 316 training rows a large grid would mostly fit noise. Performance is
reported as RMSE, MAE, and R² on the held-out test set.

In [9]:
CAT_FEATURES = ["school", "sex", "address", "famsize", "Pstatus", "Mjob", "Fjob",
                "reason", "guardian", "schoolsup", "famsup", "paid",
                "activities", "nursery", "higher", "internet", "romantic"]
NUM_BASE = ["age", "Medu", "Fedu", "traveltime", "studytime", "failures",
            "famrel", "freetime", "goout", "Dalc", "Walc", "health", "absences"]
NUM_ENG = ["parent_edu_avg", "alcohol_total", "social_index", "high_absences"]

def make_preprocessor(num_features):
    return ColumnTransformer([
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         CAT_FEATURES),
    ])

def metrics(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

def run_pipeline(df, label, use_g1g2=False):
    """Train LR, Ridge, tuned RF and tuned XGBoost; return results + best tree model."""
    num_features = NUM_BASE + NUM_ENG + (["G1", "G2"] if use_g1g2 else [])
    X = df[num_features + CAT_FEATURES]
    y = df["G3"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)
    results, fitted = {}, {}

    for name, model in [("LinearRegression", LinearRegression()),
                        ("Ridge", Ridge(alpha=1.0))]:
        pipe = Pipeline([("prep", make_preprocessor(num_features)),
                         ("model", model)])
        pipe.fit(X_train, y_train)
        results[name] = metrics(y_test, pipe.predict(X_test))
        fitted[name] = pipe

    # Note: the estimators run single-threaded (n_jobs=1) while the search
    # itself parallelises (n_jobs=-1) — nesting the two oversubscribes the CPU
    # and slows everything down on a small dataset like this.
    rf_grid = {"model__n_estimators": [100, 200, 300],
               "model__max_depth": [None, 5, 10, 15],
               "model__min_samples_split": [2, 5],
               "model__min_samples_leaf": [1, 2],
               "model__max_features": ["sqrt", None]}
    rf = Pipeline([("prep", make_preprocessor(num_features)),
                   ("model", RandomForestRegressor(random_state=42, n_jobs=1))])
    rf_search = RandomizedSearchCV(rf, rf_grid, n_iter=12, cv=3,
                                   scoring="neg_root_mean_squared_error",
                                   random_state=42, n_jobs=-1)
    rf_search.fit(X_train, y_train)
    results["RandomForest"] = metrics(y_test, rf_search.predict(X_test))
    fitted["RandomForest"] = rf_search.best_estimator_
    print(f"{label} — best RF params: {rf_search.best_params_}")

    xgb_grid = {"model__n_estimators": [100, 200],
                "model__max_depth": [3, 5, 7],
                "model__learning_rate": [0.05, 0.1, 0.2],
                "model__subsample": [0.8, 1.0],
                "model__colsample_bytree": [0.8, 1.0]}
    xgb = Pipeline([("prep", make_preprocessor(num_features)),
                    ("model", XGBRegressor(random_state=42, n_jobs=1))])
    xgb_search = RandomizedSearchCV(xgb, xgb_grid, n_iter=12, cv=3,
                                    scoring="neg_root_mean_squared_error",
                                    random_state=42, n_jobs=-1)
    xgb_search.fit(X_train, y_train)
    results["XGBoost"] = metrics(y_test, xgb_search.predict(X_test))
    fitted["XGBoost"] = xgb_search.best_estimator_
    print(f"{label} — best XGB params: {xgb_search.best_params_}")

    rows = []
    for name, (rmse, mae, r2) in results.items():
        rows.append({"Model": name, "RMSE": round(rmse, 3),
                     "MAE": round(mae, 3), "R²": round(r2, 3)})
        tag = "with_g1g2" if use_g1g2 else "no_g1g2"
        print(f"RESULT|{label}|{tag}|{name}|{rmse:.3f}|{mae:.3f}|{r2:.3f}")
    table = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
    best = table.iloc[0]
    print(f"BEST|{label}|{'with_g1g2' if use_g1g2 else 'no_g1g2'}|"
          f"{best['Model']}|{best['RMSE']:.3f}|{best['MAE']:.3f}|{best['R²']:.3f}")
    return table, fitted, X_test, y_test

In [10]:
# Main analysis: mathematics, WITHOUT G1/G2
mat_table, mat_fitted, X_test_mat, y_test_mat = run_pipeline(mat, "math")
print()
print(mat_table.to_string(index=False))

math — best RF params: {'model__n_estimators': 100, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__max_features': None, 'model__max_depth': 10}


math — best XGB params: {'model__subsample': 0.8, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.8}
RESULT|math|no_g1g2|LinearRegression|4.181|3.395|0.147
RESULT|math|no_g1g2|Ridge|4.179|3.394|0.148
RESULT|math|no_g1g2|RandomForest|3.901|3.124|0.258
RESULT|math|no_g1g2|XGBoost|3.965|3.148|0.233


BEST|math|no_g1g2|RandomForest|3.901|3.124|0.258

           Model  RMSE   MAE    R²
    RandomForest 3.901 3.124 0.258
         XGBoost 3.965 3.148 0.233
           Ridge 4.179 3.394 0.148
LinearRegression 4.181 3.395 0.147


**Reading the table.** The best model on the held-out test set is **RandomForest** (RMSE 3.90, MAE 3.12, R² 0.26). Against the Linear Regression baseline (RMSE 4.18, R² 0.15), the tuned tree models improve on the linear fit, which suggests some non-linear structure that the linear models miss — but with only 79 test students, small gaps between models should not be over-read.

In [11]:
# Feature importance from the best tree model on the mathematics data
tree_scores = {m: mat_table.loc[mat_table["Model"] == m, "RMSE"].iloc[0]
               for m in ["RandomForest", "XGBoost"]}
best_tree = min(tree_scores, key=tree_scores.get)
pipe = mat_fitted[best_tree]
feature_names = [n.split("__", 1)[1] for n in
                 pipe.named_steps["prep"].get_feature_names_out()]
importances = pipe.named_steps["model"].feature_importances_
imp = (pd.DataFrame({"feature": feature_names, "importance": importances})
       .sort_values("importance", ascending=False).head(12))

fig, ax = plt.subplots(figsize=(9, 5.5))
sns.barplot(data=imp, x="importance", y="feature", ax=ax)
ax.set_title(f"Top 12 features — {best_tree} (mathematics, without G1/G2)")
ax.set_xlabel("Feature importance")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("figures/feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()
top3 = imp.head(3)
print(f"Best tree model: {best_tree} (test RMSE {tree_scores[best_tree]:.3f})")
print("TOP3|" + "; ".join(f"{r.feature}|{r.importance:.3f}"
                          for r in top3.itertuples()))
print(imp.to_string(index=False))

Best tree model: RandomForest (test RMSE 3.901)
TOP3|absences|0.205; failures|0.159; health|0.053
            feature  importance
      absences    0.205174
      failures    0.159187
        health    0.052609
           age    0.040384
     studytime    0.032117
         goout    0.030879
  social_index    0.029684
parent_edu_avg    0.029383
        famrel    0.021461
      freetime    0.021399
    traveltime    0.020900
 alcohol_total    0.017213


![Feature importance](figures/feature_importance.png)

The RandomForest model leans most heavily on absences (0.20), failures (0.16), and health (0.05). Past failures is the strongest single correlate in the EDA and ranks second here, which is a good consistency check. Absences topping the list is the more interesting result: its linear correlation with grades is near zero, so the trees are exploiting the non-monotonic pattern (zero-absence students scoring lowest) that a correlation cannot see.

## 5. What if we add G1 and G2?

G1 and G2 are the student's grades in the first and second periods of the same
subject. They are, unsurprisingly, extremely strong predictors of the final
grade — but using them to "predict" G3 is close to circular: by the time G2 is
known, most of the school year has passed. The comparison below quantifies
exactly how much they add.

In [12]:
# Same pipeline, but WITH G1 and G2 included
mat_g12_table, _, _, _ = run_pipeline(mat, "math", use_g1g2=True)
print()
print(mat_g12_table.to_string(index=False))

math — best RF params: {'model__n_estimators': 200, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__max_features': None, 'model__max_depth': 15}


math — best XGB params: {'model__subsample': 0.8, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.8}
RESULT|math|with_g1g2|LinearRegression|2.391|1.657|0.721
RESULT|math|with_g1g2|Ridge|2.385|1.650|0.723
RESULT|math|with_g1g2|RandomForest|2.003|1.192|0.804
RESULT|math|with_g1g2|XGBoost|1.987|1.167|0.807
BEST|math|with_g1g2|XGBoost|1.987|1.167|0.807

           Model  RMSE   MAE    R²
         XGBoost 1.987 1.167 0.807
    RandomForest 2.003 1.192 0.804
           Ridge 2.385 1.650 0.723
LinearRegression 2.391 1.657 0.721


Adding G1 and G2 lifts the best model (XGBoost) to R² 0.81 (RMSE 1.99), up from R² 0.26 without them. That jump — roughly 55 extra percentage points of variance explained — is exactly why they are excluded from the main models: earlier grades in the same subject are near-proxies for the final grade, so a model that uses them is mostly restating what is already known. The no-G1/G2 model is the honest one for an early-warning use case.

## 6. Comparison: Portuguese language

The same pipeline (without G1/G2) is run on the Portuguese dataset to check
whether the findings generalise to a different subject with a larger sample
(649 students). The pipeline, split, and tuning are identical.

In [13]:
por_table, _, _, _ = run_pipeline(por, "por")
print()
print(por_table.to_string(index=False))

por — best RF params: {'model__n_estimators': 200, 'model__min_samples_split': 2, 'model__min_samples_leaf': 1, 'model__max_features': 'sqrt', 'model__max_depth': 15}


por — best XGB params: {'model__subsample': 0.8, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.8}
RESULT|por|no_g1g2|LinearRegression|2.880|2.179|0.150
RESULT|por|no_g1g2|Ridge|2.876|2.174|0.152
RESULT|por|no_g1g2|RandomForest|2.775|1.995|0.211
RESULT|por|no_g1g2|XGBoost|2.804|2.056|0.194
BEST|por|no_g1g2|RandomForest|2.775|1.995|0.211

           Model  RMSE   MAE    R²
    RandomForest 2.775 1.995 0.211
         XGBoost 2.804 2.056 0.194
           Ridge 2.876 2.174 0.152
LinearRegression 2.880 2.179 0.150


On Portuguese, the best model is **RandomForest** (RMSE 2.77, MAE 2.00, R² 0.21), compared with R² 0.26 for mathematics. Portuguese grades vary less (standard deviation 3.2 vs 4.6), so the lower RMSE is partly a smaller target to miss; R² is the fairer comparison, and on that basis the two subjects behave similarly — background and behaviour carry real but limited signal in both.

## 7. Findings

**What predicts student grades in this data?**

- **Past failures dominate; study time helps modestly.** Students with past class failures score markedly lower (11.3 with none, 5.7 with three), and grades rise modestly with weekly study time (10.1 to 11.3). The best model's three most important features are absences (0.20), failures (0.16), and health (0.05).
- **Parental education matters.** Mean grades rise steadily with both parents' education level (setting aside the 2–3 students whose parents have no formal education, too few to average meaningfully), and the engineered parental-education average is among the stronger predictors.
- **Behaviour shows up in the model more than in simple correlations.** Absences tops the feature-importance list despite a near-zero linear correlation, because the pattern is non-monotonic. Alcohol use and social time show only weak negative associations (−0.06 and −0.08), and the high-absence flag is essentially uncorrelated with grades.
- **None of the tested group differences is significant.** None of study time (ANOVA F = 1.73, p = 0.161), school (Welch's t-test, p = 0.343), or guardian (ANOVA, p = 0.205) reaches significance in this sample — the group gaps in the charts are consistent with chance.
- **Extra support looks negative — but that is selection.** Students receiving
  school support score lower on average because support goes to students who
  are already struggling, not because support harms grades.

**Model performance (mathematics, without G1/G2).** The best model was
**RandomForest**, with a test RMSE of 3.90, MAE of 3.12,
and R² of 0.26. In plain terms: its predictions are typically within
about 3.12 grade points of the true final grade, and it explains about
26% of the variation in grades.

**The honest caveat.** Background and behaviour factors alone explain only part
of the variance — most of what determines a final grade (prior knowledge,
teaching quality, motivation on the day, and the earlier grades themselves) is
not in this dataset. Adding G1 and G2 lifts R² to 0.81, which shows how
much signal sits in earlier grades — and why the no-G1/G2 model is the honest
one for an early-warning use case, where the point is to flag at-risk students
*before* exam results exist.

**Portuguese comparison.** On the Portuguese dataset the same pipeline's best model (RandomForest) reached R² 0.21 (RMSE 2.77, MAE 2.00) — broadly consistent with mathematics, suggesting the findings are not specific to one subject.